# Lab 3: Cleaning I (Missing, Wrong, Duplicated)

**DSA 405 · Week 3**

| | |
|---|---|
| **In class** | Friday, Sep 4 |
| **A3 due** | Thursday, Sep 10, 11:59 PM |
| **File** | `wolfpack_dining_raw.csv` |
| **Time** | ~25 min in class, ~55 min at home |

## Overview

Lab 2 profiled the dining file and flagged columns not to trust. This Lab acts on
those findings and records every action taken. Each cleaning decision discards
information; the cleaning log states which, with counts.

The technical core is one idea in three forms: a value can be present, look plausible,
and still be wrong.

In [42]:
# ---------------------------------------------------------------------------
# DSA 405 setup
# ---------------------------------------------------------------------------
import pandas as pd, numpy as np, requests, io

DATA = "https://raw.githubusercontent.com/jon-holt/DSA-405-Student/main/datasets/"
# DATA = "data/raw/"          # local users


def load(filename, kind="csv", **kw):
    """Read a class file whether DATA is a URL or a local folder."""
    path = DATA + filename
    if kind == "csv":
        return pd.read_csv(path, **kw)
    if kind == "excel":
        return pd.read_excel(path, **kw)
    if kind == "text":
        return requests.get(path, timeout=30).text if path.startswith("http") else open(path).read()
    if kind == "json":
        if path.startswith("http"):
            return requests.get(path, timeout=30).json()
        import json as _j
        return _j.load(open(path))
    raise ValueError(kind)


pd.set_option("display.width", 160)
print("pandas", pd.__version__)

pandas 2.2.3


---
# Part 1: Explore (in class)

## Task 1.1: The sentinel census, and the two zeros

A **sentinel** is a special value written inside a data column to signal "no real value here," instead of leaving the cell empty.

Read everything as strings first (Lab 2 showed why). Then count what the `score` and
`seats` columns contain:

In [43]:
dining = load("wolfpack_dining_raw.csv", dtype=str)

print("--- score ---")
print(dining.score.str.strip().value_counts(dropna=False).head(8))
print()
print("--- seats ---")
print(dining.seats.str.strip().value_counts(dropna=False).head(8))

--- score ---
score
100.0    14
89.8     11
92.0      9
NaN       8
0         7
88.5      7
87.8*     6
96.3      6
Name: count, dtype: int64

--- seats ---
seats
8       23
2       20
6       20
7       19
1       15
0       14
-999    13
4       12
Name: count, dtype: int64


Two findings:

- `seats` contains **13** values of `-999` and **14** zeros
- `score` contains **7** zeros

The digit `0` appears in both columns. In one it is a legitimate value; in the other it
is impossible and must mean "not recorded." Deciding which is which, with evidence, is
Checkpoint question 1. Consider what each column measures: a food truck genuinely seats
nobody.

## Task 1.2: Type repair

`score` does not convert cleanly: some values carry whitespace, some a trailing `*`
(the file's way of marking revised scores), some the word `unknown`. Repair in steps
and check each step:

In [44]:
s = dining.score.str.strip().str.rstrip("*")
score = pd.to_numeric(s, errors="coerce")

print("failed to convert (now NaN):", score.isna().sum())
print("min:", score.min(), "| max:", score.max(), "| mean:", round(score.mean(), 1))

failed to convert (now NaN): 17
min: 0.0 | max: 100.0 | mean: 90.6


Note the minimum: `min = 0.0`. The conversion succeeded, and that is the problem. The
seven zero-scores passed straight through `to_numeric` because they are valid numbers.
`errors="coerce"` catches text that fails to parse; it cannot catch a valid number that
is not a real measurement. That check has to be written explicitly:

In [45]:
score_clean = score.replace(0, np.nan)

print(f"zeros converted to NaN: {(score == 0).sum()}")
print("min is now:", score_clean.min(), "| mean:", round(score_clean.mean(), 1))

zeros converted to NaN: 7
min is now: 81.5 | mean: 92.4


## Task 1.3: Exact duplicates

Some rows appear in the file twice, identical in every column:

In [46]:
n_dup = dining.duplicated().sum()
print(f"exact duplicate rows: {n_dup}")

deduped = dining.drop_duplicates()
print(f"{len(dining)} rows in, {len(deduped)} out, {len(dining) - len(deduped)} dropped as exact duplicates")

exact duplicate rows: 46
366 rows in, 320 out, 46 dropped as exact duplicates


The last print line has the shape of a cleaning log entry: the action taken, the
number of rows affected, stated so a reader could re-derive it. Every cleaning action
for the rest of the course gets an entry in that form.

---
## Checkpoint: submit before leaving class

1. The two zeros: in which column is `0` legitimate, in which is it a disguised missing
   value, and what is the evidence?

The 0 in the seats value is legit, while the 0 in the score value is disguised as a missing value, because it is illilogical that a dining score would be zero. Instead, it was likely inteneded to be a missing value.

2. How many exact duplicate rows are in the file, and what would keeping them do to a mean?

There are 46 exact duplicates in the file, and that skews the mean more towards those duplicated values.

3. What should `-999` in `seats` become, and why?

"-999" in "seats" should become a NaN value, because -999 is not a logical number of seats and is being used as a placeholder for a missing value.





---
# Part 2: A3 (Cleaning I)

Graded. Four tasks. Task 2.4 carries the most rubric weight.

## Task 2.1: Sentinel policy, defended

Decide the fate of every sentinel found in `score` and `seats`: the zeros, the
`-999`s, the text values. For each: convert it, keep it, or flag it, with one sentence
of justification per decision. "Zero seats is real for food trucks" is a justification;
"converted everything to NaN" is not.

In [47]:
print(dining.score.unique())
print(dining.seats.unique())

score_s = dining.score.str.strip().str.rstrip("*")
score_clean = pd.to_numeric(score_s, errors="coerce").replace(0, np.nan)

seats_s = dining.seats.str.strip()
seats_clean = pd.to_numeric(seats_s, errors="coerce").replace(-999, np.nan)


['100.0' '90.1' '  ' '93.1' '86.8' '92.4' '92.0' '91.6' '89.3' '96.4'
 '81.7' 'unknown' '92.9' '89.8' '90.7' '90.5' '85.8' '86.4' ' 98.1 '
 '91.3' '91.1' '97.4' '87.7' '92.2' '87.8*' '93.8' '90.0' '91.7' '90.8'
 '91.0' '97.1' '89.4' '97.7' '94.6' '87.1' '99.7' nan '87.8' '93.9' '0'
 '94.5' '98.5' '89.7' '90.6' '85.7' '97.5' '87.5' '92.1' '93.6*' '81.5'
 '96.2' '92.6' '98.8' '97.8' '91.8' '94.7*' '93.4' '95.4' '89.2' '96.9'
 '88.6' '93.2' '87.3' '94.8' '95.5' '91.5*' ' 95.6 ' '97.0' '96.0' '99.8'
 '89.5' '94.1' '93.3' '93.0' '93.6' '90.9' ' 86.0 ' '87.9' '84.6' '96.3'
 '88.2' '90.4' '98.9' '97.3' '89.6' '96.8' '88.0' '88.5' '91.2' '88.4'
 ' 96.6 ' '94.7' '91.5' '96.9*' '90.3' '87.2' '94.3' '89.1' '91.9' '97.2'
 '94.4' '95.0' '98.1' '95.1' '95.8' '99.9' '98.4' '95.2' '88.7' '88.8'
 '86.7' '100.0*' '87.6' ' 95.4 ' '87.0' '93.7' '86.9' '98.0*' '86.0'
 '97.9' '95.7' '92.3' '99.3' '93.5' '89.0' '89.7*' '92.7*' '88.3' '95.6'
 ' 86.1 ' '98.0' '94.0' '96.7' '92.5' '91.0*' '98.6' '86.3']
['28' '

For scores, ' ' and 'unknown' are converted to NaN because they don't represent any actual score. 0 is also converted to NaN because the realistic scores in this range are around 80-100. The values with an aterisk need to be flagged, because they represent actual plausible scores, but they asterisk could mean something. Values with extra spaces should be kept but with the extra whitespace stripped.

For seats, 'unknown', ' ', and -999 are converted to NaN because they don't represent realistic values for seats. 0 is kept as a real value because it lies within the realistic range. 'nan' is kept as NaN.

## Task 2.2: A ranking that reverses

Compute mean `score` by category twice, once treating the zero-scores as real values
and once as missing, and compare the rankings.

The categories first need consolidating (27 strings, ~6 real categories, profiled in
Lab 2). Building that mapping properly is Week 4's job, so this week it is provided for you:

In [48]:
def canon(s):
    """Quick-and-dirty category canonicalizer. Week 4 teaches you to build this."""
    if pd.isna(s):
        return s
    s = " ".join(s.strip().lower().replace("-", " ").split())
    return {"c store": "convenience", "convenience store": "convenience",
            "coffee shop": "coffee", "café": "cafe",
            "fastcasual": "fast casual", "foodtruck": "food truck"}.get(s, s)

dining["category_clean"] = dining.category.map(canon)
print(dining.category_clean.value_counts())

dining["category_clean"] = dining.category.map(canon)

category_clean
food truck     107
coffee          57
fast casual     53
cafe            51
dining hall     50
convenience     48
Name: count, dtype: int64


In [49]:
score_with_zero = pd.to_numeric(
    dining.score.str.strip().str.rstrip("*"),
    errors="coerce"
)

dining["score_with_zero"] = score_with_zero

dining["score_clean"] = score_clean

rank_with_zero = dining.groupby("category_clean").score_with_zero.mean().sort_values(ascending=False)
rank_without_zero = dining.groupby("category_clean").score_clean.mean().sort_values(ascending=False)

print("Mean with zros treated as real:")
print(rank_with_zero)

print("\nMean with zeros treated as missing values:")
print(rank_without_zero)

Mean with zros treated as real:
category_clean
dining hall    94.708333
cafe           93.731111
coffee         93.041509
fast casual    92.139583
food truck     90.245794
convenience    80.004167
Name: score_with_zero, dtype: float64

Mean with zeros treated as missing values:
category_clean
dining hall    94.708333
cafe           93.731111
convenience    93.663415
coffee         93.041509
fast casual    92.139583
food truck     90.245794
Name: score_clean, dtype: float64


Report the two means for the category that moves most, and both rankings. State which
category ranks worst under each treatment. Add two sentences on how a decision made in
Week 3 (sentinel policy) changes a conclusion that would be presented in Week 14.

The category that moved the most was convenience, which was at 80 while zeros were counted as real values ranking 6th, and 93.7 when the zeroes were converted to missing values ranking 3rd. In the first treatment counting the zeroes as real values, convenience ranks worst, while in the second cleaned treatment, food truck ranks first. A change in sentinal policy caused the ranking of mean scores of different dining categories to fluctuate. In week fourteen, this would lead to a different conclusion about what the best dining options are on campus.

## Task 2.3: Duplicates the naive check misses

`drop_duplicates()` with no arguments only catches rows that match exactly, byte for
byte. `"Port City Java "` and `"port city java"` are the same place with different case
and whitespace.

1. Normalize `location_name` (strip, lowercase, collapse internal whitespace) into a new
   column.
2. Count duplicates on `(unit_code, normalized name, inspection_date)`.
3. Report: exact duplicates, duplicates after normalizing, and how many the naive check
   missed. Show two rows that only the normalized check catches.

In [50]:
location_name_clean = dining.location_name.str.strip().str.lower().str.split().str.join(" ")
dining["location_name_clean"] = location_name_clean


exact_duplicates = dining.duplicated().sum()

normalized_duplicates = dining.duplicated(subset=["unit_code", "location_name_clean", "inspection_date"]).sum()

missed = normalized_duplicates - exact_duplicates

print("Exact duplicates:", exact_duplicates)
print("Duplicates after normalizing:", normalized_duplicates)
print("Missed by naive check:", missed)

normalized = dining.duplicated(
    subset=["unit_code", "location_name_clean", "inspection_date"],
    keep=False
)

exact = dining.duplicated(keep=False)

dining.loc[
    normalized & ~exact,
    ["unit_code", "location_name", "location_name_clean", "inspection_date"] #This code helps me find rows that I can use to demonstrate
].head(30)

print("\nTwo rows that are duplicates only after the normalized check:")

dining.loc[[14, 16], ["unit_code", "location_name", "location_name_clean", "inspection_date"]]



Exact duplicates: 46
Duplicates after normalizing: 66
Missed by naive check: 20

Two rows that are duplicates only after the normalized check:


,unit_code,location_name,location_name_clean,inspection_date
14,6285,PULLEN MARKET,pullen market,1/6/25
16,6285,Pullen Market,pullen market,1/6/25


## Task 2.4: Start the cleaning log

Assemble everything A3 did into a cleaning log: a markdown
table with one row per action.

| # | Action | Rows affected | Rows remaining | Why |
|---|---|---|---|---|

Requirements:

- **8 to 12 rows**, covering every change made (conversions, NaN decisions, drops)
- Every row carries a **count**, and the arithmetic must close: rows-in minus drops equals
  rows-out, exactly. Numbers that do not reconcile mean an action went unlogged; find it.
- The **Why** column is one specific clause. "Impossible value for this measurement"
  qualifies; "cleaning" does not.

This table is the seed of the P2 cleaning log (due Oct 1), where the same standard
applies to your own project data.

*Log here.*

| # | Action | Rows affected | Rows remaining | Why |
|---|---|---|---|---|

| 1 | Stripped leading and trailing whitespace from entires in score. | 19 rows affected | 366 rows remaining | Extra whitespace causes inconsistent score formatting. |

| 2 | Striped the ending "*" from entries in score. | 20 rows affected | 366 rows remaining | Trailing asterisks prevent numeric score operations. |

| 3 | Converted "unknown" entries in score to NaN. | 5 rows affected | 366 rows remaining | "unknown" indicates a missing score value. |

| 4 | Converted blank entries in score to NaN. | 4 rows affected | 366 rows remaining | Blank entries in score do not contain any information. |

| 5 | Convrted zero entries in score to NaN. | 7 rows affected | 366 rows remaining | Zero entries in score are illogical and skew numerical conversions because the range for rest of the score values was betewen 80 - 100. |

| 6 | Converted "unknown" entries in seats to NaN. | 4 rows affected | 366 rows remaining | "unknown" indicates a mising seat count value. |

| 7 | Converted blank entries in seats to NaN. | 2 rows affected | 366 rows remaining | Blank entries in seats do not contain any information. |

| 8 | Converted "-999" entries in seats to NaN. | 13 rows affected | 366 rows remaining | Seat counts of "-999" are impossible. |

| 9 | Standardized entries in category. | 268 rows affected | 366 rows remaining | The formatting for different labels referring to the same category was made consistent. |

| 10 | Normalized entries in location_name by stripping whitespace, converting to lowercase, and collapsing internal spaces. | 366 rows affected | 366 rows remaining | Made the formatting in location_name consistent to correctly identify duplicates. |









In [51]:
print("Rows in:", len(dining))

print("Score whitespace:", (dining.score != dining.score.str.strip()).sum())
print("Score with *:", dining.score.str.strip().str.endswith("*", na=False).sum())
print("Score unknown:", dining.score.str.strip().eq("unknown").sum())
print("Score blank:", dining.score.str.strip().eq("").sum())
print("Score zeros:", dining.score.str.strip().eq("0").sum())

print("Seats unknown:", dining.seats.str.strip().eq("unknown").sum())
print("Seats blank:", dining.seats.str.strip().eq("").sum())
print("Seats -999:", dining.seats.str.strip().eq("-999").sum())

print(
    "Categories changed:",
    (dining.category.fillna("NA") != dining.category_clean.fillna("NA")).sum()
)

print(
    "Location names changed:",
    (dining.location_name.fillna("NA") !=
     dining.location_name_clean.fillna("NA")).sum()
)

print("Exact duplicates found:", dining.duplicated().sum())
print("Rows out:", len(dining))

Rows in: 366
Score whitespace: 19
Score with *: 20
Score unknown: 5
Score blank: 4
Score zeros: 7
Seats unknown: 4
Seats blank: 2
Seats -999: 13
Categories changed: 268
Location names changed: 366
Exact duplicates found: 46
Rows out: 366


---
## AI use note

List any AI tools used and what they were used for. If none, write "none." One or two
sentences.

I used chatgpt in task 2.2 to calculate group to calculate the mean of dining scores grouped by category names. I used chatgpt in task 2.3 to calculate duplicates of unit_code, normalized name, and inspection date and then again to list the rows that were only counted as duplicates in the normalized checks. I used chatgpt in task 2.4 to determine how many rows were affected by each of my actions throughout A3.

---
## Submitting

1. **Runtime > Restart runtime**, then **Run all**.
2. `File > Download > Download .ipynb`
3. Rename to `DSA405_002_FA26_A3_[yourUnityID].ipynb`
4. Upload to the **A3** space on Moodle.

The **Checkpoint** section is submitted separately to **Week 3 In-Class Activity**, before
the end of class on Friday. Due for A3: **Thursday, Sep 10, 11:59 PM**.